# AI Real Estate Agent - EDA and Training Notebook

This notebook documents how the sklearn pipeline shipped in this app was selected and trained.

It covers:
- dataset loading and quick EDA checks,
- the 12 selected features used by the local app,
- training and evaluation through the sklearn pipeline, and
- prompt-version comparison results from `prompt_logs/comparison.json`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from app.ml.pipeline import DEFAULT_DATA_PATH, SELECTED_FEATURES, TARGET, run_training


## 1. Load the dataset

The local app uses the Kaggle Ames Housing `train.csv` file stored under `data/train.csv`.


In [ ]:
data_path = ROOT / DEFAULT_DATA_PATH
df = pd.read_csv(data_path)
df.head()


## 2. Quick EDA for the selected features

The project keeps the feature set presentation-friendly: the model uses 12 columns that are both predictive and realistically describable in plain language.


In [ ]:
eda = pd.DataFrame({
    'dtype': df[SELECTED_FEATURES + [TARGET]].dtypes.astype(str),
    'missing_pct': (df[SELECTED_FEATURES + [TARGET]].isna().mean() * 100).round(2),
})
eda


In [ ]:
numeric_cols = [col for col in SELECTED_FEATURES if pd.api.types.is_numeric_dtype(df[col])]
df[numeric_cols + [TARGET]].corr(numeric_only=True)[[TARGET]].sort_values(TARGET, ascending=False)


## 3. Train the pipeline

This calls the exact project training module used by the app and Docker image. The pipeline handles imputation, scaling, encoding, model training, validation selection, and single-shot test evaluation.


In [ ]:
training_summary = run_training()
training_summary


## 4. Prompt comparison evidence

Stage 1 has two prompt variants (`v1` straightforward, `v2` few-shot). The benchmark in `prompt_logs/comparison.json` runs both against the same 5 labeled cases and picks a winner by extraction accuracy.

In [ ]:
comparison_path = ROOT / 'prompt_logs' / 'comparison.json'
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
comparison['winner'], comparison['winner_reason']


In [ ]:
pd.DataFrame(comparison['summary']).T


## 5. Design notes

Things worth knowing when reading this code:
- the split happens before fitting any preprocessing statistics,
- the sklearn `ColumnTransformer` keeps training-only medians/modes inside the saved pipeline,
- Stage 1 is validated with Pydantic and falls back safely on malformed output, and
- the UI pauses after extraction so the user can review or fill missing features before prediction runs.